# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library. The FAIR² dataset describes clinicopathological and molecular characteristics of second primary colorectal cancer (CRC) in cancer survivors, including MSI-H status and anatomical distribution.

### Dataset Source
The dataset is accessible via a Croissant schema URL, defining the structure and metadata in JSON-LD.

In [ ]:
# Ensure `mlcroissant` library (Python >=3.8) is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and explore its top-level details using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset and obtain high-level metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset: {getattr(metadata, 'name', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")

## 2. Data Overview
Review record sets, their fields, and column `@id`s.

We will enumerate record sets and fields, referencing their `@id` for reliable access. This is key for robust data operations using the Croissant standard.

In [ ]:
# List record sets and their fields, referencing by @id
record_sets = dataset.record_sets
if not record_sets:
    print("No explicit record sets defined in metadata.")
else:
    print(f"Total record sets: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"- RecordSet name: {getattr(rs, 'name', 'N/A')} | @id: {getattr(rs, '@id', 'N/A')}")
        print("  Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"    - Field name: {getattr(field, 'name', 'N/A')} | @id: {getattr(field, '@id', 'N/A')}")
            if hasattr(field, 'columns'):
                for col in field.columns:
                    print(f"      - Column name: {getattr(col, 'name', 'N/A')} | @id: {getattr(col, '@id', 'N/A')}")
        print()

# If record_sets is empty, try to iterate available data via records()
if not record_sets:
    print("Attempting to list available data records directly...")
    # Try to infer available record set ids from generator
    try:
        preview = []
        for rec in dataset.records():
            preview.append(rec)
            if len(preview) >= 2:
                break
        print("Sample records (showing up to 2):")
        for i, rec in enumerate(preview):
            print(f"Record {i+1}: {rec}")
    except Exception as e:
        print(f"Failed to preview records: {e}")


## 3. Data Extraction
Load the primary tabular record set into a DataFrame. This uses the record set and field `@id` identifiers for durable code.

- If there are record sets, we will load data using their `@id`.
- If no named record sets exist, we will attempt loading records with the default method.

In [ ]:
dataframes = {}

if dataset.record_sets:
    # Extract available record set @id
    record_set_ids = [getattr(rs, '@id', None) for rs in dataset.record_sets]
    print(f"Available record sets by @id: {record_set_ids}")

    for record_set_id in record_set_ids:
        # Load records into a pandas DataFrame
        try:
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded record set {record_set_id}. Shape: {dataframes[record_set_id].shape}")
        except Exception as e:
            print(f"Could not load {record_set_id}: {e}")

    if record_set_ids:
        main_rs_id = record_set_ids[0]
        print(f"\nSample columns for record set {main_rs_id}:")
        print(dataframes[main_rs_id].columns.tolist())
        display(dataframes[main_rs_id].head())
else:
    # If no explicit record sets exist, try to load directly
    print("No record sets defined. Attempting to load default records...")
    try:
        default_records = list(dataset.records())
        dataframes['default'] = pd.DataFrame(default_records)
        print(dataframes['default'].columns.tolist())
        display(dataframes['default'].head())
    except Exception as e:
        print(f"Could not load records: {e}")

## 4. Exploratory Data Analysis (EDA)
In this section, we'll perform basic EDA steps:
* Filtering records based on a numeric field
* Normalizing numeric values
* Grouping by a categorical field

All field and column accesses use their `@id` for stable references. You can adapt the field selection below according to your exploration needs.

In [ ]:
# Choose which DataFrame to use
if dataset.record_sets and dataframes:
    # Use the first record set by default for analysis
    main_rs_id = list(dataframes.keys())[0]
    df = dataframes[main_rs_id]
else:
    main_rs_id = 'default'
    df = dataframes.get('default', pd.DataFrame())

print(f"Using DataFrame from record set: {main_rs_id}\nShape: {df.shape}")
print(f"Column names: {df.columns.tolist()}")

# Attempt to identify a numeric field for demonstration:
# We'll heuristically select the first field containing 'age', 'interval', 'time', or 'years', or a numeric dtype.
numeric_field = None
for col in df.columns:
    lower = str(col).lower()
    if any(keyword in lower for keyword in ['age', 'interval', 'years', 'time']):
        numeric_field = col
        break
if numeric_field is None and not df.empty:
    # Fallback: first numeric column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns
    if len(numeric_candidates):
        numeric_field = numeric_candidates[0]

if numeric_field is None:
    print("No obvious numeric field found for filtering/normalizing.")
else:
    print(f"Numeric field selected for EDA: {numeric_field}")
    # Ensure type conversion
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Drop NA for demonstration
    valid_df = df.dropna(subset=[numeric_field])

    # Set a threshold as an example (median or absolute)
    threshold = valid_df[numeric_field].median() if not valid_df.empty else 0
    filtered_df = valid_df[valid_df[numeric_field] > threshold]

    print(f"Filtered records where {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / 
        filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 0
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Look for a sensible grouping field (e.g., 'sex', 'msi', 'anatomical', 'status')
    group_field = None
    for col in df.columns:
        lower = str(col).lower()
        if any(word in lower for word in ['sex', 'msi', 'category', 'anatom', 'group', 'status', 'comorb']):
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found for group-by demonstration.")

## 5. Visualization
Let's plot basic distribution(s) of the chosen numeric field and comparison(s) by group, if available.

In [ ]:
if numeric_field is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    plt.hist(filtered_df[numeric_field].dropna(), bins=10, alpha=0.7)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'{numeric_field} distribution (filtered, > median)')
    plt.show()

    if group_field is not None and group_field in filtered_df.columns:
        box_df = filtered_df[[numeric_field, group_field]].dropna()
        plt.figure(figsize=(8, 4))
        box_df.boxplot(column=numeric_field, by=group_field, grid=False)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
We've demonstrated programmatic exploration of the FAIR² colorectal cancer dataset as defined by its Croissant schema. Key steps included robust loading via `mlcroissant`, dynamically referencing fields and record sets by their `@id`, preliminary EDA, and visualizations.

This workflow can be extended for hypothesis testing, statistical modeling, or building machine learning pipelines, leveraging the standardized schema provided by Croissant.